In [5]:
#importing the libery
import pandas as pd
import sqlite3
from scipy.stats import chi2_contingency


In [6]:
# creating new data base for clean file
def convert_csv_into_db(customers, transactions, db_name):
    conn = sqlite3.connect(db_name)
    df1 = pd.read_csv(customers)
    df1.to_sql('customers_clean' , conn , if_exists = 'replace' , index=False )

    df2 = pd.read_csv(transactions)
    df2.to_sql('transactions_clean', conn , if_exists = 'replace' , index=False)
    #connections close
    conn.close()

In [7]:
open('customers_clean.csv','r').close()
open('transactions_clean.csv','r').close()

convert_csv_into_db('customers_clean.csv', 'transactions_clean.csv', 'ecommerce_clean.db')


In [8]:
conn = sqlite3.connect('ecommerce_clean.db')

table = pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", conn)
print(table)

                 name
0     customers_clean
1  transactions_clean


In [9]:
#checking if cancel/refund is caused by payment_method

In [10]:
query = """ SELECT payment_method, COUNT(*) as total_tx, SUM(CASE WHEN transaction_status IN('Cancelled','Refunded') THEN 
    1 ELSE 0 END) as bad_tx FROM transactions_clean GROUP BY payment_method"""

payment_summary = pd.read_sql(query,conn)

In [11]:
print(payment_summary)

  payment_method  total_tx  bad_tx
0  Bank Transfer      1654     219
1           Cash      1642     218
2    Credit Card      1670     213
3     Debit Card      1577     185
4       E-wallet      1656     202


In [12]:
payment_summary['bad_per'] = round(100 * payment_summary['bad_tx']/payment_summary['total_tx'],2)
payment_summary['good_tx'] = payment_summary['total_tx'] - payment_summery['bad_tx']
payment_summary['good_per'] = round(100 * payment_summary['good_tx']/payment_summary['total_tx'],2)


NameError: name 'payment_summery' is not defined

In [ ]:
# Chi-square test: is the spread in cancel/refund rate across payment methods
# real, or could it just be random chance given our sample sizes?
contingency_table = payment_summary[['bad_tx', 'good_tx']].values
chi2, p_value, dof, expected = chi2_contingency(contingency_table)
print("p-value:", p_value)


In [ ]:
#checking using discount_missing

In [ ]:
query = """ SELECT discount_missing_flag, COUNT(*) as total_tx , SUM(CASE WHEN transaction_status IN('Cancelled','Refunded')
        THEN 1 ELSE 0 END) as bad_tx from transactions_clean GROUP BY discount_missing_flag"""

discount_summary = pd.read_sql(query,conn)
print(discount_summary)

In [ ]:
discount_summary['good_tx'] = discount_summary['total_tx'] - discount_summary['bad_tx']
contingency_table = discount_summary[['bad_tx', 'good_tx']].values


In [ ]:
chi2, p_value, dof, expected = chi2_contingency(contingency_table)
print("p-value:", p_value)



In [ ]:
#checking using age
query = """ SELECT c.age_missing, COUNT(*) as total_tx , SUM(CASE WHEN t.transaction_status IN('Cancelled','Refunded')
        THEN 1 ELSE 0 END) AS bad_tx FROM customers_clean c JOIN transactions_clean t ON c.customer_id = t.customer_id  
        GROUP BY c.age_missing"""

age_summary = pd.read_sql(query,conn)
print(age_summary)

In [ ]:
age_summary['good_tx'] = age_summary['total_tx'] - age_summary['bad_tx']
contingency_table = age_summary[['bad_tx', 'good_tx']].values
chi2, p_value, dof, expected = chi2_contingency(contingency_table)
print("p-value:", p_value)

In [ ]:
#checking using state

In [ ]:
query = """ SELECT c.state_missing, COUNT(*) as total_tx , SUM(CASE WHEN t.transaction_status IN('Cancelled','Refunded')
        THEN 1 ELSE 0 END) AS bad_tx FROM customers_clean c JOIN transactions_clean t ON c.customer_id = t.customer_id  
        GROUP BY c.state_missing"""

state_summary = pd.read_sql(query,conn)
print(state_summary)

In [ ]:
state_summary['good_tx'] = state_summary['total_tx'] - state_summary['bad_tx']
contingency_table = state_summary[['bad_tx', 'good_tx']].values
chi2, p_value, dof, expected = chi2_contingency(contingency_table)
print("p-value:", p_value)

In [ ]:
#checking useing subscribe

In [ ]:
#finding value contained by subscribe
print(pd.read_sql("SELECT DISTINCT subscribe FROM customers_clean", conn))

In [ ]:
query = """
SELECT c.subscribe,
       COUNT(*) AS total_tx,
       SUM(CASE WHEN t.transaction_status IN ('Cancelled','Refunded') THEN 1 ELSE 0 END) AS bad_tx
FROM customers_clean c
JOIN transactions_clean t ON c.customer_id = t.customer_id
GROUP BY c.subscribe
"""
subscribe_summary = pd.read_sql(query, conn)
print(subscribe_summary)

In [ ]:
subscribe_summary['good_tx'] = subscribe_summary['total_tx'] - subscribe_summary['bad_tx']
contingency_table = subscribe_summary[['bad_tx', 'good_tx']].values
chi2, p_value, dof, expected = chi2_contingency(contingency_table)
print("p-value:", p_value)

In [ ]:
#checking trend of reveune vs time for timepass

In [ ]:
print(pd.read_sql("PRAGMA table_info(transactions_clean)", conn))

In [ ]:
query = """
SELECT strftime('%Y-%m', transaction_date) AS month,
       COUNT(*) AS tx_count,
       SUM(quantity * unit_price) AS gross_revenue,
       SUM(CASE WHEN transaction_status IN ('Cancelled','Refunded') THEN quantity * unit_price ELSE 0 END) AS lost_revenue
FROM transactions_clean
GROUP BY month
ORDER BY month
"""
monthly_summary = pd.read_sql(query, conn)
print(monthly_summary)

In [ ]:
monthly_summary['lost_pct'] = round(100 * monthly_summary['lost_revenue'] / monthly_summary['gross_revenue'], 2)
print(monthly_summary)

In [ ]:
# using state wise name if there any difference

In [13]:
query = """
SELECT c.state,
       COUNT(*) AS total_tx,
       SUM(CASE WHEN t.transaction_status IN ('Cancelled','Refunded') THEN 1 ELSE 0 END) AS bad_tx
FROM customers_clean c
JOIN transactions_clean t ON c.customer_id = t.customer_id
GROUP BY c.state
ORDER BY total_tx DESC
"""

                        state  total_tx  bad_tx
0                     Unknown       447      66
1              Kepulauan Riau       304      45
2                      Banten       304      48
3   Kepulauan Bangka Belitung       292      44
4              Sumatera Utara       282      37
5                  Jawa Timur       282      34
6              Sumatera Barat       275      28
7                       Papua       272      38
8           Sulawesi Tenggara       267      26
9            Kalimantan Timur       267      35
10                Jawa Tengah       267      29
11             Sulawesi Utara       266      29
12        Nusa Tenggara Barat       266      37
13                       Riau       248      28
14                Papua Barat       248      19
15               Maluku Utara       248      18
16                       Aceh       237      37
17                       Bali       228      20
18                  Gorontalo       227      18
19                   Bengkulu       222 

In [14]:
state_by_value = pd.read_sql(query, conn)
state_by_value['bad_pct'] = round(100 * state_by_value['bad_tx'] / state_by_value['total_tx'], 2)
print(state_by_value.sort_values('bad_pct', ascending=False))

                        state  total_tx  bad_tx  bad_pct
33             Sulawesi Barat       134      24    17.91
25                DKI Jakarta       209      37    17.70
24           Kalimantan Utara       211      37    17.54
2                      Banten       304      48    15.79
20                    Lampung       217      34    15.67
16                       Aceh       237      37    15.61
32                      Jambi       148      23    15.54
3   Kepulauan Bangka Belitung       292      44    15.07
1              Kepulauan Riau       304      45    14.80
0                     Unknown       447      66    14.77
7                       Papua       272      38    13.97
12        Nusa Tenggara Barat       266      37    13.91
31         Kalimantan Selatan       157      21    13.38
21              DI Yogyakarta       217      29    13.36
27                 Jawa Barat       188      25    13.30
26                     Maluku       196      26    13.27
23          Kalimantan Tengah  

In [15]:
state_by_value['good_tx'] = state_by_value['total_tx'] - state_by_value['bad_tx']
contingency_table = state_by_value[['bad_tx', 'good_tx']].values
from scipy.stats import chi2_contingency
chi2, p_value, dof, expected = chi2_contingency(contingency_table)
print("p-value:", p_value)

p-value: 0.008724729892562563


In [16]:
query = """
SELECT c.state,
       COUNT(*) AS total_tx,
       SUM(CASE WHEN t.transaction_status IN ('Cancelled','Refunded') THEN 1 ELSE 0 END) AS bad_tx
FROM customers_clean c
JOIN transactions_clean t ON c.customer_id = t.customer_id
WHERE c.state != 'Unknown'
GROUP BY c.state
ORDER BY total_tx DESC
"""
state_by_value_clean = pd.read_sql(query, conn)

In [17]:
state_by_value_clean['bad_pct'] = round(100 * state_by_value_clean['bad_tx'] / state_by_value_clean['total_tx'], 2)
print(state_by_value_clean.sort_values('bad_pct', ascending=False))

                        state  total_tx  bad_tx  bad_pct
32             Sulawesi Barat       134      24    17.91
24                DKI Jakarta       209      37    17.70
23           Kalimantan Utara       211      37    17.54
1                      Banten       304      48    15.79
19                    Lampung       217      34    15.67
15                       Aceh       237      37    15.61
31                      Jambi       148      23    15.54
2   Kepulauan Bangka Belitung       292      44    15.07
0              Kepulauan Riau       304      45    14.80
6                       Papua       272      38    13.97
11        Nusa Tenggara Barat       266      37    13.91
30         Kalimantan Selatan       157      21    13.38
20              DI Yogyakarta       217      29    13.36
26                 Jawa Barat       188      25    13.30
25                     Maluku       196      26    13.27
22          Kalimantan Tengah       213      28    13.15
3              Sumatera Utara  

In [18]:
state_by_value_clean['good_tx'] = state_by_value_clean['total_tx'] - state_by_value_clean['bad_tx']
contingency_table = state_by_value_clean[['bad_tx', 'good_tx']].values
from scipy.stats import chi2_contingency
chi2, p_value, dof, expected = chi2_contingency(contingency_table)
print("p-value:", p_value)

p-value: 0.009084920046042453
